# Fitting: Computing an NLL

We will be using JAX this time. JAX gives you the NumPy API plus function
transformations: `jit` compiles a function with XLA, `grad` differentiates it,
and `vmap` vectorizes it. Like before, let's set up the data first:

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

dist = np.hstack(
    [
        rng.normal(loc=1, scale=2.0, size=500_000),
        rng.normal(loc=1, scale=0.5, size=500_000),
    ]
)

Now let's load JAX. By default JAX uses 32-bit floats, even if you hand it a
64-bit array, and silently downcasts. We turn 64-bit on so the comparison with
the other notebooks is fair:

In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

print(f"{jax.__version__ = }")
jax.devices()

Arrays are placed on the default device, which is the GPU if one is visible.
Converting the dataset moves it to the GPU:

In [ ]:
d_dist = jnp.asarray(dist)
d_dist.devices()

This looks almost exactly like the NumPy version; only the namespace changed.
JAX arrays are immutable, but we never assign into one here:

In [ ]:
def gaussian(x, μ, σ):
    return 1 / jnp.sqrt(2 * jnp.pi * σ**2) * jnp.exp(-((x - μ) ** 2) / (2 * σ**2))


def add(x, f_0, mean, sigma, sigma2):
    return f_0 * gaussian(x, mean, sigma) + (1 - f_0) * gaussian(x, mean, sigma2)


def nll(dist, f_0, mean, sigma, sigma2):
    return -jnp.sum(jnp.log(add(dist, f_0, mean, sigma, sigma2)))

JAX dispatches asynchronously, like CuPy, so we call `.block_until_ready()`
before the timer stops. Without `jit`, every operation is a separate kernel
launch:

In [ ]:
%%timeit
nll(d_dist, *rng.random(4)).block_until_ready()

Now let's compile the whole function with XLA. The first call traces the
Python and compiles; later calls with the same shapes and dtypes reuse the
cached executable. This could also be written as a decorator, `@jax.jit`:

In [ ]:
nll_jit = jax.jit(nll)

In [ ]:
%%time
nll_jit(d_dist, *rng.random(4)).block_until_ready()

In [ ]:
%%timeit
nll_jit(d_dist, *rng.random(4)).block_until_ready()

XLA fuses the elementwise work and the reduction into a few kernels, which is
what we did by hand with CuPy's `ElementwiseKernel`.

## Gradients

The other reason to use JAX is `grad`. It transforms a function into one that
returns the gradient with respect to the arguments you pick (here the four
parameters, not the data). It composes with `jit`:

In [ ]:
nll_grad = jax.jit(jax.grad(nll, argnums=(1, 2, 3, 4)))
nll_grad(d_dist, 0.5, 1.0, 2.0, 0.5)

In [ ]:
%%timeit
jax.block_until_ready(nll_grad(d_dist, *rng.random(4)))

This is what a minimizer wants: the value and the gradient in one compiled
call (`jax.value_and_grad` gives both).

## Exercise

Turn off `jax_enable_x64` (restart the kernel; it must be set before any array
is made) and run again in 32-bit. How does the performance compare? Which of
the inputs are still 64-bit, and what does JAX do with them?